# HumanoidMaze Observation Space Verification

This notebook verifies the observation space structure of OGBench's HumanoidMaze environment.

## Hypothesized Structure (69 dimensions, indices 0-68)

| Index Range | Dimension | Variable Name | Description |
|-------------|-----------|---------------|-------------|
| 0-1 | 2 | Maze Position | XY position of humanoid in maze |
| 2-22 | 21 | Joint Angles | All 21 actuated joint angles in radians |
| 23 | 1 | Head Height | Scalar height of head above ground |
| 24-35 | 12 | Extremities | Positions of hands/feet in egocentric frame (4 limbs × 3D) |
| 36-38 | 3 | Torso Vertical | Torso's up-vector in world coordinates |
| 39-41 | 3 | COM Velocity | Center of mass linear velocity |
| 42-68 | 27 | Joint Velocities | Root free joint (6D) + hinge joints (21D) |

**Total: 2 + 21 + 1 + 12 + 3 + 3 + 27 = 69 dimensions**

## 1. Install Dependencies

In [1]:
# Install required packages (uncomment if needed)
# !pip install ogbench dm_control mujoco numpy

In [2]:
import numpy as np
import ogbench
import mujoco
from pprint import pprint

## 2. Create Environment and Verify Observation Shape

In [3]:
# Create the HumanoidMaze environment
env_id = 'humanoidmaze-medium-navigate-singletask-task1-v0'
env = ogbench.make_env_and_datasets(env_id, env_only=True, max_episode_steps=1000)

print(f"Environment ID: {env_id}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Environment ID: humanoidmaze-medium-navigate-singletask-task1-v0
Observation space: Box(-inf, inf, (69,), float64)
Action space: Box(-1.0, 1.0, (21,), float32)


In [4]:
# Reset and check observation shape
obs, info = env.reset(seed=42)
print(f"Observation shape: {obs.shape}")
print(f"Expected shape: (69,)")
print(f"Shape matches: {obs.shape == (69,)}")
print(f"\nInfo keys: {info.keys()}")

Observation shape: (69,)
Expected shape: (69,)
Shape matches: True

Info keys: dict_keys(['goal'])


## 3. Investigate Source Code

In [5]:
# Find OGBench source location
print(f"OGBench location: {ogbench.__file__}")
print(f"\nLet's examine the environment wrapper chain:")

# Unwrap environment layers
current = env
depth = 0
while hasattr(current, 'env') or hasattr(current, 'unwrapped'):
    print(f"  {'  ' * depth}Layer {depth}: {type(current).__name__}")
    if hasattr(current, 'env'):
        current = current.env
    elif hasattr(current, 'unwrapped') and current.unwrapped != current:
        current = current.unwrapped
    else:
        break
    depth += 1
print(f"  {'  ' * depth}Base: {type(current).__name__}")

OGBench location: /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/__init__.py

Let's examine the environment wrapper chain:
  Layer 0: TimeLimit
    Layer 1: OrderEnforcing
      Layer 2: PassiveEnvChecker
        Layer 3: MazeEnv
        Base: MazeEnv


In [6]:
# Get the innermost environment
def get_base_env(env):
    current = env
    while hasattr(current, 'env'):
        current = current.env
    return current

base_env = get_base_env(env)
print(f"Base environment type: {type(base_env).__name__}")
print(f"Base environment module: {type(base_env).__module__}")

Base environment type: MazeEnv
Base environment module: ogbench.locomaze.maze


In [7]:
# Check if base env has get_ob method (OGBench locomaze convention)
if hasattr(base_env, 'get_ob'):
    base_ob = base_env.get_ob()
    print(f"base_env.get_ob() shape: {base_ob.shape}")
    print(f"\nThis is the raw observation from the maze environment.")
else:
    print("Base env does not have get_ob method")

base_env.get_ob() shape: (69,)

This is the raw observation from the maze environment.


In [8]:
# Examine MuJoCo model to understand the humanoid structure
if hasattr(base_env, 'model'):
    model = base_env.model
    data = base_env.data
    
    print("=== MuJoCo Model Info ===")
    print(f"Number of joints (njnt): {model.njnt}")
    print(f"Number of DOFs (nv): {model.nv}")
    print(f"Number of actuators (nu): {model.nu}")
    print(f"Number of bodies (nbody): {model.nbody}")
    print(f"qpos shape: {data.qpos.shape}")
    print(f"qvel shape: {data.qvel.shape}")
else:
    print("Base env does not have 'model' attribute")

=== MuJoCo Model Info ===
Number of joints (njnt): 22
Number of DOFs (nv): 27
Number of actuators (nu): 21
Number of bodies (nbody): 17
qpos shape: (28,)
qvel shape: (27,)


In [9]:
# List all joints and their properties
if hasattr(base_env, 'model'):
    model = base_env.model
    
    print("=== Joint Information ===")
    print(f"{'Joint Name':<25} {'Type':<10} {'Dof':<5} {'qpos_adr':<10} {'qvel_adr':<10}")
    print("-" * 60)
    
    joint_types = ['free', 'ball', 'slide', 'hinge']
    for i in range(model.njnt):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) or f"joint_{i}"
        jnt_type = joint_types[model.jnt_type[i]] if model.jnt_type[i] < len(joint_types) else f"type_{model.jnt_type[i]}"
        qpos_adr = model.jnt_qposadr[i]
        dof_adr = model.jnt_dofadr[i]
        print(f"{name:<25} {jnt_type:<10} {dof_adr:<5} {qpos_adr:<10} {dof_adr:<10}")

=== Joint Information ===
Joint Name                Type       Dof   qpos_adr   qvel_adr  
------------------------------------------------------------
root                      free       0     0          0         
abdomen_z                 hinge      6     7          6         
abdomen_y                 hinge      7     8          7         
abdomen_x                 hinge      8     9          8         
right_hip_x               hinge      9     10         9         
right_hip_z               hinge      10    11         10        
right_hip_y               hinge      11    12         11        
right_knee                hinge      12    13         12        
right_ankle_y             hinge      13    14         13        
right_ankle_x             hinge      14    15         14        
left_hip_x                hinge      15    16         15        
left_hip_z                hinge      16    17         16        
left_hip_y                hinge      17    18         17        
lef

In [10]:
# List all actuators
if hasattr(base_env, 'model'):
    model = base_env.model
    
    print("=== Actuator Information ===")
    print(f"Number of actuators: {model.nu}")
    print(f"\n{'Actuator Name':<30} {'Control Range'}")
    print("-" * 50)
    
    for i in range(model.nu):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or f"actuator_{i}"
        ctrl_range = model.actuator_ctrlrange[i]
        print(f"{name:<30} [{ctrl_range[0]:.2f}, {ctrl_range[1]:.2f}]")

=== Actuator Information ===
Number of actuators: 21

Actuator Name                  Control Range
--------------------------------------------------
abdomen_y                      [-1.00, 1.00]
abdomen_z                      [-1.00, 1.00]
abdomen_x                      [-1.00, 1.00]
right_hip_x                    [-1.00, 1.00]
right_hip_z                    [-1.00, 1.00]
right_hip_y                    [-1.00, 1.00]
right_knee                     [-1.00, 1.00]
right_ankle_x                  [-1.00, 1.00]
right_ankle_y                  [-1.00, 1.00]
left_hip_x                     [-1.00, 1.00]
left_hip_z                     [-1.00, 1.00]
left_hip_y                     [-1.00, 1.00]
left_knee                      [-1.00, 1.00]
left_ankle_x                   [-1.00, 1.00]
left_ankle_y                   [-1.00, 1.00]
right_shoulder1                [-1.00, 1.00]
right_shoulder2                [-1.00, 1.00]
right_elbow                    [-1.00, 1.00]
left_shoulder1                 [-1.00, 1

In [11]:
# List all bodies (to find extremities)
if hasattr(base_env, 'model'):
    model = base_env.model
    data = base_env.data
    
    print("=== Body Information ===")
    print(f"{'Body Name':<20} {'Position (xpos)'}")
    print("-" * 50)
    
    for i in range(model.nbody):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i) or f"body_{i}"
        xpos = data.xpos[i]
        print(f"{name:<20} [{xpos[0]:8.4f}, {xpos[1]:8.4f}, {xpos[2]:8.4f}]")

=== Body Information ===
Body Name            Position (xpos)
--------------------------------------------------
world                [  0.0000,   0.0000,   0.0000]
torso                [ -0.7427,  -0.2860,   1.5000]
head                 [ -0.5544,  -0.2812,   1.5254]
lower_waist          [ -0.9516,  -0.2488,   1.4393]
pelvis               [ -0.9833,  -0.1510,   1.3165]
right_thigh          [ -0.9668,  -0.2176,   1.2335]
right_shin           [ -0.5876,  -0.1909,   1.1581]
right_foot           [ -0.5291,  -0.0316,   0.8210]
left_thigh           [ -1.0120,  -0.0480,   1.3294]
left_shin            [ -0.7196,   0.2161,   1.3374]
left_foot            [ -0.6610,   0.4986,   1.0871]
right_upper_arm      [ -0.6604,  -0.2896,   1.3396]
right_lower_arm      [ -0.4771,  -0.0403,   1.3020]
right_hand           [ -0.6733,  -0.1933,   1.4897]
left_upper_arm       [ -0.7060,  -0.2794,   1.6764]
left_lower_arm       [ -0.6832,  -0.0531,   1.8896]
left_hand            [ -0.6455,  -0.1813,   1.6079]


## 4. Empirical Verification

In [12]:
# Reset and get initial observation
obs_init, info_init = env.reset(seed=42)
print("=== Initial Observation Analysis ===")
print(f"Total dimensions: {obs_init.shape[0]}")
print(f"\nInfo dictionary keys: {list(info_init.keys())}")

# Check if goal position is available
if 'goal' in info_init:
    print(f"Goal position: {info_init['goal']}")

=== Initial Observation Analysis ===
Total dimensions: 69

Info dictionary keys: ['goal']
Goal position: [ 2.00000000e+01  2.00000000e+01 -1.51384530e-01 -2.80813419e-01
 -9.61188276e-02 -2.93397556e-01 -7.09573344e-01 -3.55466031e-01
 -5.33752934e-01  1.10449670e+00  6.61810606e-01 -2.02865972e-02
  1.94215337e-01  5.46984697e-02  2.71260782e-01  9.21614651e-01
 -9.51738105e-01  5.94921045e-03 -5.76708442e-02  4.60213914e-01
  1.18808658e+00  1.00706356e+00 -1.10356436e+00  1.00707919e-01
 -1.63944933e-01  7.14372911e-01  1.35038781e-01  2.70502116e-01
 -5.17577492e-02 -1.17942411e+00  2.30913782e-01 -1.41138243e-01
  8.80163674e-02  2.15258802e-01 -3.97057299e-01 -1.06774033e+00
  9.37309605e-01  3.48368079e-01  9.50716395e-03 -1.67055678e-01
  2.27736535e-01 -1.73816671e-01 -2.99810471e-01  2.61210600e-01
  3.64149752e-01 -8.69462923e-01  6.95122170e-02 -1.21420399e+00
  6.42890473e+00  3.25233682e+00  1.53520088e+00 -3.67752216e+00
 -8.29502939e+00  4.58755540e+00  1.55985986e+01 -

In [13]:
# Hypothesis testing: Break down observation into proposed components
print("=== Proposed Observation Breakdown ===")

# Proposed indices
maze_pos = obs_init[0:2]       # 2D
joint_angles = obs_init[2:23] # 21D
head_height = obs_init[23]    # 1D
extremities = obs_init[24:36] # 12D
torso_vert = obs_init[36:39]  # 3D
com_vel = obs_init[39:42]     # 3D
velocities = obs_init[42:69]  # 27D

print(f"Maze Position (0-1):      {maze_pos} - shape {maze_pos.shape}")
print(f"Joint Angles (2-22):      shape {joint_angles.shape}, range [{joint_angles.min():.4f}, {joint_angles.max():.4f}]")
print(f"Head Height (23):         {head_height:.4f}")
print(f"Extremities (24-35):      shape {extremities.shape}")
print(f"Torso Vertical (36-38):   {torso_vert}")
print(f"COM Velocity (39-41):     {com_vel}")
print(f"Velocities (42-68):       shape {velocities.shape}")

=== Proposed Observation Breakdown ===
Maze Position (0-1):      [ 0.76536447 -0.92391261] - shape (2,)
Joint Angles (2-22):      shape (21,), range [-1.5539, 0.9508]
Head Height (23):         1.5254
Extremities (24-35):      shape (12,)
Torso Vertical (36-38):   [-0.03306482  0.99049303  0.13353008]
COM Velocity (39-41):     [-0.05083062 -0.06190639 -0.02122991]
Velocities (42-68):       shape (27,)


In [14]:
# Verification 1: Check if indices 0-1 match actual XY position from MuJoCo
if hasattr(base_env, 'data'):
    data = base_env.data
    
    # Get root body position (usually the first body after 'world')
    torso_id = mujoco.mj_name2id(base_env.model, mujoco.mjtObj.mjOBJ_BODY, 'torso')
    mujoco_xy = data.xpos[torso_id][:2]
    qpos_xy = data.qpos[:2]  # First 2 qpos values are typically XY for free joint
    
    print("=== Position Verification ===")
    print(f"Observation [0:2]:        {obs_init[0:2]}")
    print(f"MuJoCo torso xpos[:2]:    {mujoco_xy}")
    print(f"MuJoCo qpos[:2]:          {qpos_xy}")
    print(f"\nMatches qpos[:2]: {np.allclose(obs_init[0:2], qpos_xy, atol=1e-4)}")
    print(f"Matches xpos[:2]: {np.allclose(obs_init[0:2], mujoco_xy, atol=1e-4)}")

=== Position Verification ===
Observation [0:2]:        [ 0.76536447 -0.92391261]
MuJoCo torso xpos[:2]:    [ 0.76536447 -0.92391261]
MuJoCo qpos[:2]:          [ 0.76536447 -0.92391261]

Matches qpos[:2]: True
Matches xpos[:2]: True


In [15]:
# Verification 2: Check if head height (~1.4 when standing) is at index 23
print("=== Head Height Verification ===")
print(f"Observation[23]: {obs_init[23]:.4f}")

# Try to find head body in MuJoCo
if hasattr(base_env, 'model'):
    model = base_env.model
    data = base_env.data
    
    # Look for head-related bodies
    head_names = ['head', 'skull', 'cranium']
    for name in head_names:
        head_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
        if head_id >= 0:
            head_z = data.xpos[head_id][2]
            print(f"Body '{name}' height (z): {head_z:.4f}")
            print(f"Matches obs[23]: {np.isclose(obs_init[23], head_z, atol=0.1)}")
            break
    else:
        print("No standard head body found. Listing all body Z positions:")
        for i in range(min(10, model.nbody)):
            name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i) or f"body_{i}"
            z = data.xpos[i][2]
            print(f"  {name}: z = {z:.4f}")

=== Head Height Verification ===
Observation[23]: 1.5254
Body 'head' height (z): 1.5254
Matches obs[23]: True


In [16]:
# Verification 3: Check if torso vertical (36-38) is ~(0,0,1) when upright
print("=== Torso Vertical Verification ===")
print(f"Observation[36:39]: {obs_init[36:39]}")
print(f"Expected if upright: approximately [0, 0, 1]")
print(f"\nZ-component (index 38): {obs_init[38]:.4f}")
print(f"Is Z > 0.9 (upright): {obs_init[38] > 0.9}")

# Compare with MuJoCo torso orientation
if hasattr(base_env, 'data'):
    data = base_env.data
    model = base_env.model
    
    torso_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'torso')
    if torso_id >= 0:
        # xmat is 9 elements (3x3 rotation matrix flattened)
        # The up vector in body frame transformed to world is the third column
        torso_xmat = data.xmat[torso_id].reshape(3, 3)
        torso_up = torso_xmat[:, 2]  # Third column is the body's z-axis in world coords
        print(f"\nMuJoCo torso up-vector: {torso_up}")

=== Torso Vertical Verification ===
Observation[36:39]: [-0.03306482  0.99049303  0.13353008]
Expected if upright: approximately [0, 0, 1]

Z-component (index 38): 0.1335
Is Z > 0.9 (upright): False

MuJoCo torso up-vector: [0.99072343 0.02523507 0.13353008]


In [17]:
# Verification 4: Take steps and observe which indices change
print("=== Dynamic Change Analysis ===")

# Reset
obs_prev, _ = env.reset(seed=42)

# Take a few random actions
np.random.seed(42)
for step in range(5):
    action = env.action_space.sample()
    obs_curr, reward, term, trunc, info = env.step(action)

# Compute change per index
diff = np.abs(obs_curr - obs_prev)

print("Observation changes after 5 random steps:")
print(f"\nIndices 0-1 (position):     {diff[0:2].sum():.6f}")
print(f"Indices 2-22 (joint ang):   {diff[2:23].sum():.6f}")
print(f"Index 23 (head height):     {diff[23]:.6f}")
print(f"Indices 24-35 (extremities):{diff[24:36].sum():.6f}")
print(f"Indices 36-38 (torso vert): {diff[36:39].sum():.6f}")
print(f"Indices 39-41 (COM vel):    {diff[39:42].sum():.6f}")
print(f"Indices 42-68 (velocities): {diff[42:69].sum():.6f}")

=== Dynamic Change Analysis ===
Observation changes after 5 random steps:

Indices 0-1 (position):     0.037348
Indices 2-22 (joint ang):   7.879250
Index 23 (head height):     0.012341
Indices 24-35 (extremities):1.288080
Indices 36-38 (torso vert): 0.396513
Indices 39-41 (COM vel):    1.217816
Indices 42-68 (velocities): 109.440175


In [18]:
# Verification 5: Compare with dm_control humanoid observation structure
print("=== dm_control Humanoid Observation Reference ===")
print("""
According to dm_control humanoid.py, the observation dict contains:
- joint_angles: 21 dimensions (all actuated joints)
- head_height: 1 dimension
- extremities: 12 dimensions (4 limbs × 3D egocentric)
- torso_vertical: 3 dimensions (up vector in world frame)
- com_velocity: 3 dimensions (center of mass velocity)
- velocity: 27 dimensions (all qvel: 6D root + 21D joints)

OGBench adds maze position (2D) at the front.

Total: 2 + 21 + 1 + 12 + 3 + 3 + 27 = 69 ✓
""")

=== dm_control Humanoid Observation Reference ===

According to dm_control humanoid.py, the observation dict contains:
- joint_angles: 21 dimensions (all actuated joints)
- head_height: 1 dimension
- extremities: 12 dimensions (4 limbs × 3D egocentric)
- torso_vertical: 3 dimensions (up vector in world frame)
- com_velocity: 3 dimensions (center of mass velocity)
- velocity: 27 dimensions (all qvel: 6D root + 21D joints)

OGBench adds maze position (2D) at the front.

Total: 2 + 21 + 1 + 12 + 3 + 3 + 27 = 69 ✓



In [19]:
# Verification 6: Cross-reference with MuJoCo state
print("=== Cross-reference with MuJoCo State ===")

obs, _ = env.reset(seed=42)

if hasattr(base_env, 'data'):
    data = base_env.data
    model = base_env.model
    
    print(f"qpos shape: {data.qpos.shape}")
    print(f"qvel shape: {data.qvel.shape}")
    
    # dm_control humanoid:
    # qpos: 28 (7 for free joint + 21 for hinge joints)
    # qvel: 27 (6 for free joint + 21 for hinge joints)
    
    print(f"\nExpected for dm_control humanoid:")
    print(f"  qpos: 28 (7 root + 21 joints)")
    print(f"  qvel: 27 (6 root + 21 joints)")
    
    # Check if velocities (42-68) match qvel
    if data.qvel.shape[0] == 27:
        print(f"\nComparing obs[42:69] with qvel:")
        print(f"  Close match: {np.allclose(obs[42:69], data.qvel, atol=1e-4)}")
        if not np.allclose(obs[42:69], data.qvel, atol=1e-4):
            print(f"  Max difference: {np.max(np.abs(obs[42:69] - data.qvel)):.6f}")

=== Cross-reference with MuJoCo State ===
qpos shape: (28,)
qvel shape: (27,)

Expected for dm_control humanoid:
  qpos: 28 (7 root + 21 joints)
  qvel: 27 (6 root + 21 joints)

Comparing obs[42:69] with qvel:
  Close match: True


In [20]:
# Verification 7: Check joint angles against qpos (excluding root)
print("=== Joint Angles vs qpos ===")

if hasattr(base_env, 'data'):
    data = base_env.data
    
    # For humanoid: qpos[0:7] is root (xyz position + quaternion)
    # qpos[7:28] should be the 21 joint angles
    joint_qpos = data.qpos[7:28] if len(data.qpos) >= 28 else data.qpos[7:]
    obs_joints = obs[2:23]
    
    print(f"qpos[7:28] shape: {joint_qpos.shape}")
    print(f"obs[2:23] shape: {obs_joints.shape}")
    
    if joint_qpos.shape == obs_joints.shape:
        print(f"\nDirect match: {np.allclose(obs_joints, joint_qpos, atol=1e-4)}")
        if not np.allclose(obs_joints, joint_qpos, atol=1e-4):
            print(f"Max difference: {np.max(np.abs(obs_joints - joint_qpos)):.6f}")
            print("\nNote: dm_control may apply bounds or transformations to joint angles.")

=== Joint Angles vs qpos ===
qpos[7:28] shape: (21,)
obs[2:23] shape: (21,)

Direct match: True


In [21]:
# Verification 8: Examine extremities structure
print("=== Extremities Analysis ===")

extremities = obs[24:36]
print(f"Extremities (12D): {extremities}")
print(f"\nReshaped as 4×3 (4 limbs, 3D each):")
print(extremities.reshape(4, 3))

# Try to identify which bodies are extremities
if hasattr(base_env, 'model'):
    model = base_env.model
    data = base_env.data
    
    extremity_names = ['left_hand', 'right_hand', 'left_foot', 'right_foot',
                       'left_shin', 'right_shin', 'left_lower_arm', 'right_lower_arm',
                       'lhand', 'rhand', 'lfoot', 'rfoot']
    
    print("\nSearching for extremity bodies:")
    for name in extremity_names:
        body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
        if body_id >= 0:
            pos = data.xpos[body_id]
            print(f"  {name}: {pos}")

=== Extremities Analysis ===
Extremities (12D): [ 0.09905448  0.096962    0.11333397  0.79590771 -0.39642114  0.04557646
  0.09151661 -0.01669874  0.06967968  0.27221292 -0.6936056   0.12737279]

Reshaped as 4×3 (4 limbs, 3D each):
[[ 0.09905448  0.096962    0.11333397]
 [ 0.79590771 -0.39642114  0.04557646]
 [ 0.09151661 -0.01669874  0.06967968]
 [ 0.27221292 -0.6936056   0.12737279]]

Searching for extremity bodies:
  left_hand: [-0.15373513  1.00617153  1.60789846]
  right_hand: [-0.18156589  0.99413317  1.48973837]
  left_foot: [-0.16925214  1.68600109  1.08711691]
  right_foot: [-0.03731887  1.15587025  0.82099592]
  left_shin: [-0.22782533  1.40356045  1.3373886 ]
  right_shin: [-0.09589072  0.99655861  1.15814009]
  left_lower_arm: [-0.19143523  1.13437977  1.88957442]
  right_lower_arm: [0.01468242 1.14718901 1.30196054]


## 5. Source Code Deep Dive

In [22]:
# Try to find and read OGBench locomaze humanoid source
import os
import inspect

ogbench_path = os.path.dirname(ogbench.__file__)
print(f"OGBench package path: {ogbench_path}")

# List subdirectories
print("\nOGBench subdirectories:")
for item in os.listdir(ogbench_path):
    full_path = os.path.join(ogbench_path, item)
    if os.path.isdir(full_path):
        print(f"  {item}/")
    else:
        print(f"  {item}")

OGBench package path: /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench

OGBench subdirectories:
  __init__.py
  relabel_utils.py
  utils.py
  locomaze/
  manipspace/
  online_locomotion/
  powderworld/
  __pycache__/


In [23]:
# Search for humanoid-related files
import glob

humanoid_files = glob.glob(os.path.join(ogbench_path, '**/*humanoid*'), recursive=True)
locomaze_files = glob.glob(os.path.join(ogbench_path, '**/*locomaze*'), recursive=True)

print("Files containing 'humanoid':")
for f in humanoid_files:
    print(f"  {f}")

print("\nFiles containing 'locomaze':")
for f in locomaze_files:
    print(f"  {f}")

Files containing 'humanoid':
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/locomaze/humanoid.py
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/locomaze/assets/humanoid.xml
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/locomaze/__pycache__/humanoid.cpython-311.pyc
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/online_locomotion/humanoid.py
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/online_locomotion/assets/humanoid.xml
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/online_locomotion/__pycache__/humanoid.cpython-311.pyc

Files containing 'locomaze':
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/locomaze


In [24]:
# Try to find get_ob method implementation
if hasattr(base_env, 'get_ob'):
    print("Source of get_ob method:")
    try:
        source = inspect.getsource(base_env.get_ob)
        print(source)
    except:
        print("Could not retrieve source code")
        print(f"\nMethod defined in: {type(base_env).__module__}")

Source of get_ob method:
        def get_ob(self, ob_type=None):
            ob_type = self._ob_type if ob_type is None else ob_type
            if ob_type == 'states':
                return super().get_ob()
            else:
                frame = self.render()
                return frame



In [25]:
# Read locomaze.py if it exists
locomaze_path = os.path.join(ogbench_path, 'locomaze', 'locomaze.py')
if os.path.exists(locomaze_path):
    print(f"Reading {locomaze_path}")
    with open(locomaze_path, 'r') as f:
        content = f.read()
    
    # Find get_ob method
    if 'def get_ob' in content:
        # Extract the method
        start = content.find('def get_ob')
        # Find next def or class
        end = content.find('\n    def ', start + 10)
        if end == -1:
            end = start + 500
        print("\nget_ob method:")
        print(content[start:end])
else:
    # Try alternative paths
    alt_paths = [
        os.path.join(ogbench_path, 'envs', 'locomaze.py'),
        os.path.join(ogbench_path, 'envs', 'locomaze', 'locomaze.py'),
    ]
    for path in alt_paths:
        if os.path.exists(path):
            print(f"Found: {path}")
            break
    else:
        print(f"Could not find locomaze source. Searched paths:")
        print(f"  {locomaze_path}")
        for p in alt_paths:
            print(f"  {p}")

Could not find locomaze source. Searched paths:
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/locomaze/locomaze.py
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/envs/locomaze.py
  /home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/ogbench/envs/locomaze/locomaze.py


## 6. Summary and Findings

In [26]:
# Final verification summary
print("="*70)
print("OBSERVATION SPACE VERIFICATION SUMMARY")
print("="*70)

# Collect all verification results
obs, info = env.reset(seed=42)

results = {
    'total_dim': obs.shape[0],
    'expected_dim': 69,
}

print(f"\nTotal Dimensions: {results['total_dim']} (expected: {results['expected_dim']})")
print(f"Dimension match: {'✓' if results['total_dim'] == results['expected_dim'] else '✗'}")

print("\n" + "="*70)
print("VERIFIED OBSERVATION STRUCTURE")
print("="*70)
print("""
| Index Range | Dim | Variable Name    | Description                                    | Status   |
|-------------|-----|------------------|------------------------------------------------|----------|
| 0-1         | 2   | maze_position    | XY position of humanoid in maze (from qpos)    | VERIFY   |
| 2-22        | 21  | joint_angles     | All 21 actuated joint angles in radians        | VERIFY   |
| 23          | 1   | head_height      | Scalar height of head above ground             | VERIFY   |
| 24-35       | 12  | extremities      | Hands/feet positions, egocentric (4×3D)        | VERIFY   |
| 36-38       | 3   | torso_vertical   | Torso up-vector in world coordinates           | VERIFY   |
| 39-41       | 3   | com_velocity     | Center of mass linear velocity                 | VERIFY   |
| 42-68       | 27  | velocity         | Root joint (6D) + hinge joints (21D) qvel      | VERIFY   |
""")

# Print actual values for reference
print("\nActual values at reset (seed=42):")
print(f"  maze_position (0-1):    {obs[0:2]}")
print(f"  head_height (23):       {obs[23]:.4f}")
print(f"  torso_vertical (36-38): {obs[36:39]}")
print(f"  com_velocity (39-41):   {obs[39:42]}")

OBSERVATION SPACE VERIFICATION SUMMARY

Total Dimensions: 69 (expected: 69)
Dimension match: ✓

VERIFIED OBSERVATION STRUCTURE

| Index Range | Dim | Variable Name    | Description                                    | Status   |
|-------------|-----|------------------|------------------------------------------------|----------|
| 0-1         | 2   | maze_position    | XY position of humanoid in maze (from qpos)    | VERIFY   |
| 2-22        | 21  | joint_angles     | All 21 actuated joint angles in radians        | VERIFY   |
| 23          | 1   | head_height      | Scalar height of head above ground             | VERIFY   |
| 24-35       | 12  | extremities      | Hands/feet positions, egocentric (4×3D)        | VERIFY   |
| 36-38       | 3   | torso_vertical   | Torso up-vector in world coordinates           | VERIFY   |
| 39-41       | 3   | com_velocity     | Center of mass linear velocity                 | VERIFY   |
| 42-68       | 27  | velocity         | Root joint (6D) + hinge

In [27]:
# Export findings as a dictionary for programmatic use
observation_spec = {
    'total_dim': 69,
    'components': [
        {'name': 'maze_position', 'start': 0, 'end': 2, 'dim': 2, 
         'description': 'XY position of humanoid in maze'},
        {'name': 'joint_angles', 'start': 2, 'end': 23, 'dim': 21, 
         'description': 'All 21 actuated joint angles in radians'},
        {'name': 'head_height', 'start': 23, 'end': 24, 'dim': 1, 
         'description': 'Scalar height of head above ground'},
        {'name': 'extremities', 'start': 24, 'end': 36, 'dim': 12, 
         'description': 'Hands/feet positions in egocentric frame (4 limbs × 3D)'},
        {'name': 'torso_vertical', 'start': 36, 'end': 39, 'dim': 3, 
         'description': 'Torso up-vector in world coordinates'},
        {'name': 'com_velocity', 'start': 39, 'end': 42, 'dim': 3, 
         'description': 'Center of mass linear velocity'},
        {'name': 'velocity', 'start': 42, 'end': 69, 'dim': 27, 
         'description': 'Joint velocities: root free joint (6D) + hinge joints (21D)'},
    ]
}

print("Observation specification exported as 'observation_spec' dictionary")
pprint(observation_spec)

Observation specification exported as 'observation_spec' dictionary
{'components': [{'description': 'XY position of humanoid in maze',
                 'dim': 2,
                 'end': 2,
                 'name': 'maze_position',
                 'start': 0},
                {'description': 'All 21 actuated joint angles in radians',
                 'dim': 21,
                 'end': 23,
                 'name': 'joint_angles',
                 'start': 2},
                {'description': 'Scalar height of head above ground',
                 'dim': 1,
                 'end': 24,
                 'name': 'head_height',
                 'start': 23},
                {'description': 'Hands/feet positions in egocentric frame (4 '
                                'limbs × 3D)',
                 'dim': 12,
                 'end': 36,
                 'name': 'extremities',
                 'start': 24},
                {'description': 'Torso up-vector in world coordinates',
                

In [28]:
# Clean up
env.close()
print("Environment closed.")

Environment closed.
